In [1]:
import psycopg
import psycopg.sql as sql
from dotenv import load_dotenv
import numpy as np
import csv
import os
import multiprocessing
from itertools import batched
from cupboard import salt_shaker

# The v1 API from authzed leaves a lot to be desired in terms of usability, but it works well enoug
from authzed.api.v1 import (
	BulkCheckPermissionRequest,
	BulkCheckPermissionRequestItem,
	InsecureClient,
	ObjectReference,
	Relationship,
	RelationshipUpdate,
	SubjectReference,
	WriteRelationshipsRequest,
	CheckPermissionRequest,
	CheckPermissionResponse,
	LookupResourcesRequest
) 

# This took a lot of searching to find. 
from authzed.api.v1.permission_service_pb2 import ImportBulkRelationshipsRequest


In [7]:
load_dotenv()

db_config = {
	'dbname': os.environ.get('DB_NAME'),
	'user': os.environ.get('DB_USER'),              
	'password': os.environ.get('DB_PASS'),    
	'host': os.environ.get('DB_HOST'),           
	'port': os.environ.get('DB_PORT'),    
	'options': '-c statement_timeout=500'        
}

RELATIONSHIPS_PER_TRANSACTION = 100
RELATIONSHIPS_PER_REQUEST_CHUNK = 10


spice_config = {
	'spice_url': os.environ.get('SPICE_URL'),
	'bearer_token': os.environ.get('SPICEDB_GRPC_PRESHARED_KEY') 
}

client = InsecureClient(
	spice_config['spice_url'], 
	spice_config['bearer_token']
)


In [3]:
data = []

with open("./generic_user_accounts_5000.csv", newline="") as f:
	reader = csv.reader(f, delimiter=',', quotechar="|") 
	header = next(reader)
	for row in reader: 
		# Ids are omitted so our database can be sure they are unique
		record = {
			"username": row[3].strip(),
			"email": row[4].strip(),
			"password": row[5].strip(),
			"created": row[6].strip(),
			"status": "invited",
			"external_auth_provider": "manual-token",
			"default_org_id": "1"
		}

		# Invalid fields are truncated
		if len(record["username"]) > 20:
			record["username"] = record["username"][0:19]
		
		data.append(record)

Password hashing is done in a parallel context to improve the throughput of the pipeline.

In [4]:
# multiprocessing behaves a little funny in notebooks. This execution guard helps. 
if __name__ == '__main__':

	jobs = list(batched(data, 209))

	with multiprocessing.Pool(processes=24) as pool:
		results = pool.map(salt_shaker, jobs)


In [5]:
insert_data = [] 

# Restructure the data from the salting pool into a form that is expected by psycopg3
for data_set in results:
	for record in data_set:
		insert_data.append(record)

In [8]:

with psycopg.connect(**db_config) as conn: 
	# A Common Table Extension is used to reduce the total number of requests to the database, improving performance. 
	# This works by allowing the previous query to act as if it were a table, meaning we can select data like ids from it.
	query_1 = sql.SQL('''
		WITH new_user as (
			INSERT INTO usermanagement.users (username, email, password_hash, status, default_org_id, created)
			VALUES (%(username)s, %(email)s, %(password_hash)s, %(status)s, %(default_org_id)s, %(created)s)
			RETURNING user_id, default_org_id
		), orgsert AS (
			INSERT INTO usermanagement.organizations_users (organization_id, user_id, role_id)
			SELECT default_org_id, user_id, 2
			FROM new_user
			
			RETURNING user_id
		)
			INSERT INTO projectmanagement.projects_users (project_id, user_id, role_id)
			SELECT 1, user_id, 2
			FROM orgsert;
		
	''')

	with conn.cursor() as curr:
		# Executemany is a method exposed by psycopg3 that allows you to efficently batch a large number of queries. This significantly
		# improves performance.
		curr.executemany(query_1, insert_data)

In [9]:
with psycopg.connect(**db_config) as conn:
	# Having the users in the database is only half the story, since our ReBAC layer will deny them access to any data. So 
	# we must pull them back out and inform spiceDB about these users and what they're allowed to do.
	query_1 = sql.SQL('''
		SELECT uuid FROM usermanagement.users WHERE user_id != 0 AND status = 'invited'
	''')

	query_2 = sql.SQL(' SELECT uuid from usermanagement.organizations WHERE organization_id = 1; ')


	with conn.cursor() as curr:
		user_uuids = curr.execute(query_1).fetchall()
		org_uuid = curr.execute(query_2).fetchone()[0]


In [10]:
# Psycopg outputs tuple rows by default, so we actually have a "2" 
# dimensional array that needs to be reshaped into a one dimensional array for use in the relationship generator.
u_uuids = [id_set[0] for id_set in user_uuids]

In [11]:
# Generate bulk relationship requests. This would be a good candidate for multiprocessing since its techincally embarassingly parallel, but there is really not much of a bottleneck present here. 
def relationship_generator(object_id, object_type, subject_type, subjects, relation):
    for subj in subjects:
        yield Relationship(
            resource=ObjectReference(object_type=object_type, object_id=str(object_id)),
            relation=relation,
            subject=SubjectReference(
                object=ObjectReference(object_type=subject_type, object_id=str(subj))
            ),
        )

In [12]:
# Authzed, the makers of SpiceDB suggest that large volume requests are batched to improve the throughput of building the graph 
# that tracks relationships.
org_chunks = batched(
	relationship_generator(org_uuid, 'organization', 'user', u_uuids, 'annotator'), RELATIONSHIPS_PER_TRANSACTION
)


for transaction in org_chunks:
	request_chunks = batched(transaction, RELATIONSHIPS_PER_REQUEST_CHUNK)
	response = client.ImportBulkRelationships(
					(
						ImportBulkRelationshipsRequest(relationships=relationship_chunk)
						for relationship_chunk in request_chunks
					)
				)